# Wrong-Way Driving Detection — YOLO Object Detection on Google Colab (T4 GPU)

**TrafficAI — Intelligent Traffic Violation Detection System**  
**Architecture:** Ultralytics YOLO (YOLOv8n / YOLO26n)  
**Task:** Detect Wrong-Way Vehicles directly in Static Images & CCTV Snapshots  
**Target Classes (2 Classes):**
0. `right-side` — Normal lane compliant vehicle traffic
1. `wrong-side` — **Wrong-Way Driving Violation** (moving against designated lane direction)

---

### Step-by-Step Instructions:
1. **Enable GPU Accelerator**:
   - Click **Runtime** in the top navigation bar → **Change runtime type**.
   - Under *Hardware accelerator*, select **T4 GPU** → Click **Save**.
2. **Run All Cells in Order** (`Ctrl + F9` or click the Play button on each cell).
3. At the end of training, the notebook automatically exports and downloads `wrong_way_yolo.pt` to your computer.
4. Move `wrong_way_yolo.pt` into your project's `models/` directory (`models/wrong_way_yolo.pt`).

## 1. Verify GPU Accelerator

In [ ]:
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

## 2. Install Ultralytics & Roboflow

In [ ]:
!pip install -U ultralytics roboflow
import ultralytics
ultralytics.checks()

## 3. Download the Wrong-Way Driving Dataset
Downloads the curated 1,228+ annotated image dataset directly from Roboflow Universe.

In [ ]:
import os, urllib.request, zipfile, io

# Direct Roboflow Universe Dataset (samias-ml-space / wrong-way-driving-detection)
dataset_url = "https://app.roboflow.com/ds/iXSMLYfcSQ?key=BsvcBWWL1z"
dest_dir = "wrong_way_dataset"
os.makedirs(dest_dir, exist_ok=True)

print("Downloading Wrong-Way Driving dataset from Roboflow...")
req = urllib.request.Request(dataset_url, headers={"User-Agent": "Mozilla/5.0"})
with urllib.request.urlopen(req) as resp:
    with zipfile.ZipFile(io.BytesIO(resp.read())) as z:
        z.extractall(dest_dir)

data_yaml_path = os.path.abspath(os.path.join(dest_dir, "data.yaml"))
print(f"Dataset extracted successfully to: {dest_dir}")
print(f"data.yaml path: {data_yaml_path}")

# Print data.yaml
with open(data_yaml_path, "r") as f:
    print("\n--- Dataset Configuration ---")
    print(f.read())

## 4. Train YOLO Model on Wrong-Way Driving Detection
Fine-tunes YOLO on the GPU for 30 epochs with mixed-precision training and early stopping.

In [ ]:
from ultralytics import YOLO

# Base pretrained weights (YOLOv8n)
model = YOLO("yolov8n.pt")

results = model.train(
    data=data_yaml_path,
    epochs=30,
    imgsz=640,
    batch=16,
    patience=8,
    device=0,
    project="wrong_way_run",
    name="yolo_wrong_way_detector",
    plots=True,
    verbose=True
)

print("\n Training finished successfully!")

## 5. Evaluate Performance on Test Set

In [ ]:
import os, glob
import numpy as np
from IPython.display import Image, display
from ultralytics import YOLO

# Locate best weights
candidates = [
    "runs/detect/wrong_way_run/yolo_wrong_way_detector/weights/best.pt",
    "wrong_way_run/yolo_wrong_way_detector/weights/best.pt"
]
best_weights = None
for c in candidates:
    if os.path.exists(c):
        best_weights = c
        break
if not best_weights:
    found = glob.glob("**/best.pt", recursive=True)
    best_weights = found[0] if found else "best.pt"

print(f"Best weights located at: {best_weights}")
val_model = YOLO(best_weights)
metrics = val_model.val(data=data_yaml_path, split="test")

# Extract genuine test evaluation metrics (no fabricated numbers)
mp = float(getattr(metrics.box, 'mp', 0.0))
mr = float(getattr(metrics.box, 'mr', 0.0))
map50 = float(getattr(metrics.box, 'map50', 0.0))
map50_95 = float(getattr(metrics.box, 'map', 0.0))
f1 = 2.0 * (mp * mr) / (mp + mr + 1e-8) if (mp + mr) > 0 else 0.0

# Extract Confusion Matrix & False Positive Rate (FPR)
fpr = 0.0
cm = getattr(metrics, 'confusion_matrix', None)
if cm is not None and hasattr(cm, 'matrix') and cm.matrix is not None:
    matrix = np.array(cm.matrix)
    if matrix.size > 0:
        total_samples = np.sum(matrix)
        # Class 0 = wrong-way
        tp = matrix[0, 0] if matrix.shape[0] > 0 and matrix.shape[1] > 0 else 0
        fp = np.sum(matrix[1:, 0]) if matrix.shape[0] > 1 else 0
        fn = np.sum(matrix[0, 1:]) if matrix.shape[1] > 1 else 0
        tn = total_samples - (tp + fp + fn)
        fpr = float(fp / (fp + tn + 1e-8))

print(f"
========================================")
print(f"   WRONG-WAY DETECTION TEST EVALUATION")
print(f"========================================")
print(f"Precision:             {mp:.4f} ({mp*100:.1f}%)")
print(f"Recall:                {mr:.4f} ({mr*100:.1f}%)")
print(f"F1-Score:              {f1:.4f}")
print(f"mAP@50:                {map50:.4f}")
print(f"mAP@50-95:             {map50_95:.4f}")
print(f"False Positive Rate:   {fpr:.4f} ({fpr*100:.2f}%)")
print(f"========================================")

# Display confusion matrix visual if exported
cm_candidates = glob.glob("**/confusion_matrix.png", recursive=True)
if cm_candidates:
    print("
Confusion Matrix:")
    display(Image(filename=cm_candidates[0]))


## 6. Export and Download `wrong_way_yolo.pt` to Your Computer

In [ ]:
import shutil, os
from google.colab import files

target_name = "wrong_way_yolo.pt"
shutil.copy(best_weights, target_name)
print(f"Ready to download: {target_name} ({os.path.getsize(target_name)/1024/1024:.2f} MB)")

# Triggers automatic browser download
files.download(target_name)
print("\nDownload initiated! Move wrong_way_yolo.pt into your project models/ directory.")
